In [1]:

from langchain_ollama import ChatOllama, OllamaEmbeddings

# Local Ollama chat model
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Local Ollama embedding model
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Test the model
response = llm.invoke("What is LangChain?")
print(response.content)


LangChain is an open-source, Python-based framework for building blockchain-agnostic data pipelines. It allows developers to create data pipelines that can interact with various blockchain networks, such as Ethereum, Polkadot, and Solana, as well as traditional databases.

LangChain provides a set of tools and libraries that enable developers to build data pipelines that can:

1. Read and write data to blockchain networks
2. Process and transform data using Python
3. Integrate with traditional databases and other data sources
4. Automate data workflows and pipelines

The framework is designed to be highly customizable and extensible, allowing developers to create data pipelines that meet the specific needs of their use cases.

Some key features of LangChain include:

1. Blockchain-agnostic data pipelines: LangChain allows developers to build data pipelines that can interact with multiple blockchain networks, without having to rewrite code for each network.
2. Python-based: LangChain is

In [ ]:
#parent documentretriever

In [2]:

from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import InMemoryStore
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings

# Ollama embeddings
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Sample documents
docs = [
    Document(
        page_content="LangChain helps build LLM-powered apps with memory and agents.",
        metadata={"id": "1"}
    ),
    Document(
        page_content="Agents in LangChain use tools to answer questions.",
        metadata={"id": "2"}
    )
]

# Child splitter
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

# Create an empty FAISS vectorstore
vectorstore = FAISS.from_texts(
    ["placeholder"],
    embedding
)

# Remove placeholder
vectorstore.delete(
    list(vectorstore.index_to_docstore_id.values())
)

# Parent document retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),
    child_splitter=child_splitter
)

# Add documents
retriever.add_documents(docs)

# Retrieve documents
results = retriever.invoke("What are agents?")

for doc in results:
    print("Retrieved Document:", doc.page_content)


C:\Users\gnana\AppData\Local\Temp\ipykernel_2012\929728978.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Retrieved Document: Agents in LangChain use tools to answer questions.
Retrieved Document: LangChain helps build LLM-powered apps with memory and agents.


In [ ]:
#BM25 Retriever

In [4]:

%pip install rank_bm25


Note: you may need to restart the kernel to use updated packages.


In [5]:

from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

docs = [
    Document(page_content="LangChain enables LLM applications."),
    Document(page_content="Vector search is powerful."),
    Document(page_content="BM25 is a classical retrieval method.")
]

bm25_retriever = BM25Retriever.from_documents(docs)

results = bm25_retriever.invoke(
    "How does BM25 work?"
)

for doc in results:
    print("BM25 Result:", doc.page_content)


BM25 Result: BM25 is a classical retrieval method.
BM25 Result: Vector search is powerful.
BM25 Result: LangChain enables LLM applications.


In [ ]:
##Ensemble Retriever

In [7]:

from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

# BM25 retriever
bm25_retriever = BM25Retriever.from_documents(docs)

# Ollama embeddings
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

# FAISS vector store
vectorstore = FAISS.from_documents(
    docs,
    embedding
)

# Vector retriever
vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# Ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        vector_retriever
    ],
    weights=[0.5, 0.5]
)

# Search
results = ensemble_retriever.invoke(
    "What is LangChain?"
)

for doc in results:
    print(doc.page_content)


LangChain supports LLMs.
You can build AI apps using LangChain.
Ollama runs language models locally.


In [ ]:
#Time-Weighted Vector Store Retriever

In [14]:

from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from datetime import datetime

# Ollama embeddings
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Documents
docs = [
    Document(
        page_content="LangChain is for LLM-based apps",
        metadata={"last_accessed_at": datetime.now()}
    ),
    Document(
        page_content="Vector search improves relevance",
        metadata={"last_accessed_at": datetime.now()}
    )
]

# Vectorstore
vectorstore = FAISS.from_documents(
    docs,
    embedding
)

# Time weighted retriever
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=2,
    score_threshold=None
)

# Retrieve
results = retriever.invoke(
    "What is LangChain?"
)

for doc in results:
    print(doc.page_content)


In [ ]:
#Use Ollama with your retrieved documents (RAG)

In [10]:

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Local LLM
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Local embeddings
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Documents
docs = [
    Document(
        page_content="LangChain is a framework for building applications powered by language models."
    ),
    Document(
        page_content="RAG retrieves relevant information from documents before generating an answer."
    ),
    Document(
        page_content="Ollama allows you to run language models locally."
    )
]

# Vector database
vectorstore = FAISS.from_documents(
    docs,
    embedding
)

# Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# User question
question = "What is RAG and how does it work?"

# Retrieve relevant documents
retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

# Prompt
prompt = f"""
Answer the question using only the following context.

Context:
{context}

Question:
{question}

Answer:
"""

# Generate answer using Ollama
response = llm.invoke(prompt)

print("Answer:")
print(response.content)


Answer:
RAG (Relevance-Aware Graph) is a retrieval model that retrieves relevant information from documents before generating an answer. It works by first indexing documents and their corresponding relevance scores, then using a graph-based approach to find the most relevant documents for a given query. The retrieved documents are then used as input to a language model, which generates an answer based on the information in the retrieved documents.


In [ ]:
#Use Ollama with your retrieved documents (RAG)

In [11]:

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Local LLM
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Local embeddings
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Documents
docs = [
    Document(
        page_content="LangChain is a framework for building applications powered by language models."
    ),
    Document(
        page_content="RAG retrieves relevant information from documents before generating an answer."
    ),
    Document(
        page_content="Ollama allows you to run language models locally."
    )
]

# Vector database
vectorstore = FAISS.from_documents(
    docs,
    embedding
)

# Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# User question
question = "What is RAG and how does it work?"

# Retrieve relevant documents
retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

# Prompt
prompt = f"""
Answer the question using only the following context.

Context:
{context}

Question:
{question}

Answer:
"""

# Generate answer using Ollama
response = llm.invoke(prompt)

print("Answer:")
print(response.content)


Answer:
RAG (Relevance-Aware Graph) is a retrieval model that retrieves relevant information from documents before generating an answer. It works by first indexing documents and their corresponding relevance scores, then using a graph-based approach to find the most relevant documents for a given query. The retrieved documents are then used as input to a language model, which generates an answer based on the information in the retrieved documents.


In [ ]:
#Tavily Search Retriever

In [13]:
%pip install -U tavily-python langchain-community

In [16]:

%pip install -U langchain-community langchain-ollama tavily-python


Note: you may need to restart the kernel to use updated packages.


In [17]:

import os
from getpass import getpass

from langchain_community.retrievers import TavilySearchAPIRetriever
from langchain_ollama import ChatOllama

# Tavily API key
os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API key: ")

# Ollama local LLM
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Tavily web search retriever
retriever = TavilySearchAPIRetriever(k=3)

# Search the web
query = "Latest updates about LangChain"

docs = retriever.invoke(query)

# Display retrieved documents
for i, doc in enumerate(docs, 1):
    print(f"\nDocument {i}:")
    print(doc.page_content)

# Combine search results into context
context = "\n\n".join(
    doc.page_content for doc in docs
)

# Create prompt for Ollama
prompt = f"""
You are a helpful AI assistant.

Answer the question using the following web search results.
If the information is not available in the search results,
say that you could not find it.

Web Search Results:
{context}

Question:
{query}

Answer:
"""

# Generate answer using Ollama
response = llm.invoke(prompt)

print("\nOllama Answer:")
print(response.content)


Enter Tavily API key:  ········



Document 1:
## `langchain` v1.3.0

This release adds support for `version="v3"` in `stream_events` / `astream_events` for `langchain` agents. Refer to the

event streamingguide for details.

## `langgraph` v1.2.0

This release adds finer-grained control over node execution (timeouts, error recovery, and graceful shutdown), a new channel type that cuts checkpoint overhead for long-running threads, and a new content-block-centric streaming API (v3) with typed, per-channel projections. [...] Subscribe: Our changelog includes an

RSS feedthat can integrate with

Slack,

Readybotor

RSS Feeds to Discord Bot, and other subscription tools.

## `langchain` v1.5.0

LangChain 1.5 adds decision models and decision-driven middleware. This entry also highlights recent improvements across core and integration packages since LangChain 1.4.0. [...] Pause once per tool calland the

human-in-the-loop overview.

## `langchain` v1.4.0

MCP support now ships inside LangChain in the `langchain.mcp` namespa